In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt

data_path = "../data/csv"
stats_preview = pd.read_csv(os.path.join(data_path, "other_stats.csv"), nrows=5)
games_preview = pd.read_csv(os.path.join(data_path, "game.csv"), nrows=5)



In [ ]:
stats_preview = pd.read_csv("../data/csv/other_stats.csv", nrows=5)
stats_preview

In [ ]:
games_preview.columns

In [ ]:
games_preview = pd.read_csv("../data/csv/game.csv", nrows=5)
games_preview
# step 2

In [ ]:
import pandas as pd
import os

data_path = "../data/csv"

for file in os.listdir(data_path):
    file_path = os.path.join(data_path, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(file, round(size_mb, 2), "MB")

    #step 1: load the data, this part is from here and go up, I dont know why when I try create new code blocks it is appearing from the top, and push it down to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

games = pd.read_csv("../data/csv/game.csv")

# Standardize column names
games.columns = games.columns.str.lower().str.strip().str.replace(" ", "_")

# Keep only regular season games
rows_before_filter = len(games)
regular_season_mask = (
    games["season_type"].astype("string").str.strip().str.casefold()
    == "regular season"
)
games = games.loc[regular_season_mask].copy()

print("Rows before season filter:", rows_before_filter)
print("Regular season rows:", len(games))
print("Non regular season rows removed:", rows_before_filter - len(games))

games.head()

In [ ]:
before = games.shape[0]
games = games.drop_duplicates()
after = games.shape[0]

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

# Cleaning 1: remove duplicates

In [ ]:
games["game_date"] = pd.to_datetime(games["game_date"])

games[["game_id", "game_date"]].head()

# Cleaning 2: convert date

In [ ]:
home = games[[
    "game_id", "game_date", "season_id",
    "team_id_home", "team_name_home", "wl_home",
    "pts_home", "fg_pct_home", "reb_home", "ast_home", "tov_home"
]].copy()

home.columns = [
    "game_id", "game_date", "season_id",
    "team_id", "team_name", "win_loss",
    "pts", "fg_pct", "reb", "ast", "tov"
]

home["home_away"] = "Home"


away = games[[
    "game_id", "game_date", "season_id",
    "team_id_away", "team_name_away", "wl_away",
    "pts_away", "fg_pct_away", "reb_away", "ast_away", "tov_away"
]].copy()

away.columns = [
    "game_id", "game_date", "season_id",
    "team_id", "team_name", "win_loss",
    "pts", "fg_pct", "reb", "ast", "tov"
]

away["home_away"] = "Away"


team_games = pd.concat([home, away], ignore_index=True)

team_games.head()

# Cleaning 3: create home-team rows and away-team rows

In [ ]:
team_games = team_games.sort_values(["team_id", "game_date"])

team_games.head()

# Cleaning 4: sort games chronologically by team

In [ ]:
team_games["previous_game_date"] = team_games.groupby("team_id")["game_date"].shift(1)

team_games["rest_days"] = (
    team_games["game_date"] - team_games["previous_game_date"]
).dt.days - 1

team_games[["team_name", "game_date", "previous_game_date", "rest_days"]].head(10)
# Cleaning 5: create rest_days

In [ ]:
def categorize_rest(days):
    if pd.isna(days):
        return "First Game"
    elif days <= 0:
        return "0 Days"
    elif days == 1:
        return "1 Day"
    elif days == 2:
        return "2 Days"
    else:
        return "3+ Days"

team_games["rest_category"] = team_games["rest_days"].apply(categorize_rest)

team_games["rest_category"].value_counts()

# Cleaning 6: create rest categories

In [ ]:
team_games["win"] = team_games["win_loss"].map({"W": 1, "L": 0})

team_games[["team_name", "win_loss", "win"]].head()

# Create win column

In [ ]:
summary_stats = team_games[["pts", "fg_pct", "reb", "ast", "tov", "rest_days"]].describe()

summary_stats

# EDA 1: Summary statistics

In [ ]:
eda_df = team_games[
    (team_games["rest_category"] != "First Game") &
    (team_games["rest_days"] >= 0) &
    (team_games["rest_days"] <= 7)
].copy()

plt.figure(figsize=(8, 5))
plt.hist(eda_df["rest_days"].dropna(), bins=8)
plt.title("Distribution of Rest Days")
plt.xlabel("Rest Days")
plt.ylabel("Number of Team-Games")
plt.show()

# EDA 2: Distribution of rest days


In [ ]:
points_by_rest = eda_df.groupby("rest_category")["pts"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
)

points_by_rest
# EDA 3: Average points by rest category

In [ ]:
plt.figure(figsize=(8, 5))
points_by_rest.plot(kind="bar")
plt.title("Average Points by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Average Points")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 3

In [ ]:
fg_by_rest = eda_df.groupby("rest_category")["fg_pct"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
)

fg_by_rest
# EDA 4: Shooting percentage by rest category

In [ ]:
plt.figure(figsize=(8, 5))
fg_by_rest.plot(kind="bar")
plt.title("Average Field Goal Percentage by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Average FG%")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 4

In [ ]:
win_by_rest = eda_df.groupby("rest_category")["win"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
) * 100

win_by_rest

# EDA 5: Win percentage by rest category

In [ ]:
plt.figure(figsize=(8, 5))
win_by_rest.plot(kind="bar")
plt.title("Win Percentage by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Win Percentage (%)")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 5

In [ ]:
corr = eda_df[["rest_days", "pts", "fg_pct", "reb", "ast", "tov", "win"]].corr()

corr

# EDA 6: Correlation analysis

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Between Rest Days and Performance Statistics")

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, round(corr.iloc[i, j], 2), ha="center", va="center")

plt.tight_layout()
plt.show()
# turns that table into a heatmap figure

In [ ]:
team_games.to_csv("../data/cleaned_team_rest_data.csv", index=False)

# save cleaned data

# Assignment 2: Machine Learning and Statistical Analysis

This section uses the cleaned team-game data from Assignment 1 and applies three algorithms required for CSE 487 Assignment 2: Linear Regression, Logistic Regression, and K-Means Clustering. Each algorithm is connected to the project question about whether rest affects team performance.

In [ ]:
# Assignment 2 setup
# Import math and random for algorithm implementation

import pandas as pd
import matplotlib.pyplot as plt
import random
import math

# Split data into training and testing sets
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        random.seed(random_state)
    indices = list(range(len(X)))
    random.shuffle(indices)
    test_count = int(len(indices) * test_size)
    test_indices = indices[:test_count]
    train_indices = indices[test_count:]
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]
    y_train = y.iloc[train_indices]
    y_test = y.iloc[test_indices]

    return X_train, X_test, y_train, y_test

# Evaluation metrics for regression and classification
# Mean squared error
def mean_squared_error(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    total = 0
    for actual, predicted in zip(y_true, y_pred):
        total += (actual - predicted) ** 2

    return total / len(y_true)

# R-squared score
def r2_score(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    mean_y = sum(y_true) / len(y_true)
    ss_total = 0
    ss_residual = 0
    for actual, predicted in zip(y_true, y_pred):
        ss_total += (actual - mean_y) ** 2
        ss_residual += (actual - predicted) ** 2

    return 1 - (ss_residual / ss_total)

# Accuracy score for classification
def accuracy_score(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    correct = 0
    for actual, predicted in zip(y_true, y_pred):
        if actual == predicted:
            correct += 1

    return correct / len(y_true)

# Confusion matrix
def confusion_matrix(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)

    matrix = [[0, 0], [0, 0]]

    for actual, predicted in zip(y_true, y_pred):
        matrix[int(actual)][int(predicted)] += 1

    return matrix

# Use the cleaned dataset created in Assignment 1
df = pd.read_csv("../data/cleaned_team_rest_data.csv")

# Keep only useful and realistic rest-day records
df = df[
    (df["rest_category"] != "First Game") &
    (df["rest_days"] >= 0) &
    (df["rest_days"] <= 7)
].copy()

# Keep only rows with the variables needed for the models
df = df.dropna(subset=["rest_days", "pts", "fg_pct", "reb", "ast", "tov", "win"])

print("Rows used for Assignment 2 models:", len(df))
df.head()


## Algorithm 1: Linear Regression

Linear Regression is used to test whether rest days can predict the number of points a team scores. This is useful because points scored is a continuous numerical value.

In [ ]:
# Algorithm 1: Linear Regression
# Goal: Predict team points using rest_days

X = df[["rest_days"]]
y = df["pts"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

print("Linear Regression Results")
print("Coefficient:", linear_model.coef_[0])
print("Intercept:", linear_model.intercept_)
print("R2 Score:", r2_score(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))


In [ ]:
# Visualization for Linear Regression

plt.figure(figsize=(8, 5))
plt.scatter(X_test["rest_days"], y_test, alpha=0.3, label="Actual Points")
plt.plot(X_test["rest_days"], y_pred, label="Regression Line")
plt.title("Linear Regression: Rest Days vs Team Points")
plt.xlabel("Rest Days")
plt.ylabel("Team Points")
plt.legend()
plt.show()


## Algorithm 2: Logistic Regression

Logistic Regression is used to predict a binary outcome: whether a team wins or loses. The model uses rest days and performance statistics such as field goal percentage, rebounds, assists, and turnovers.

In [ ]:
# Algorithm 2: Logistic Regression
# Goal: Predict win/loss using rest and team performance variables

features = ["rest_days", "fg_pct", "reb", "ast", "tov"]
X = df[features]
y = df["win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# Visualization for Logistic Regression: Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()

for i in range(len(cm)):
    for j in range(len(cm[i])):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xticks([0, 1], ["Loss", "Win"])
plt.yticks([0, 1], ["Loss", "Win"])
plt.show()


## Algorithm 3: K-Means Clustering

K-Means is used to group team-game performances into clusters based on similar statistical patterns. This can show whether team performances naturally separate into low, medium, and high performance groups.

In [ ]:
# Algorithm 3: K-Means Clustering
# Goal: Group team-game records into performance clusters

cluster_features = ["pts", "fg_pct", "reb", "ast", "tov"]
X = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

df[["pts", "fg_pct", "reb", "ast", "tov", "cluster"]].head()


In [ ]:
# Visualization for K-Means Clustering

plt.figure(figsize=(8, 5))
plt.scatter(df["pts"], df["fg_pct"], c=df["cluster"], alpha=0.5)
plt.title("K-Means Clustering: Points vs Field Goal Percentage")
plt.xlabel("Team Points")
plt.ylabel("Field Goal Percentage")
plt.colorbar(label="Cluster")
plt.show()


In [ ]:
# Cluster summary table

cluster_summary = df.groupby("cluster")[cluster_features + ["win", "rest_days"]].mean()
cluster_summary
